In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
 
catalog = "workspace"
schema = "stock_analytics"
 
print("=" * 70)
print("ANALYTICS LAYER - GOLD TABLES")
print("=" * 70)

In [0]:

# Cell 1: Load Silver Layer
silver_df = spark.table(f"{catalog}.{schema}.silver_stock_indicators")
 
print("✅ Loaded silver_stock_indicators")

In [0]:

# Cell 2: Gold Table 1 - Current Trading Signals
print("\n" + "=" * 70)
print("CREATING GOLD TABLES")
print("=" * 70 + "\n")
 
# Get latest price for each stock
window_latest = Window.partitionBy("Ticker").orderBy(F.desc("Date"))
 
gold_signals = silver_df \
    .withColumn("rank", F.row_number().over(window_latest)) \
    .filter(F.col("rank") == 1) \
    .select(
        "Ticker", "Date", "Close", "Volume",
        "MA_20", "MA_50", "MA_200",
        "RSI_14", "MACD_Line", "Volatility_30",
        "BB_Upper", "BB_Lower",
        "Price_Trend_20", "Price_Trend_200",
        "Golden_Cross", "Death_Cross"
    ) \
    .withColumn(
        "Trading_Signal",
        F.when(F.col("Golden_Cross") == "BUY_SIGNAL", "STRONG_BUY")
         .when(F.col("Death_Cross") == "SELL_SIGNAL", "STRONG_SELL")
         .when((F.col("RSI_14") < 30) & (F.col("Price_Trend_20") == "Bullish"), "BUY")
         .when((F.col("RSI_14") > 70) & (F.col("Price_Trend_20") == "Bearish"), "SELL")
         .when((F.col("MACD_Line") > F.col("MACD_Signal")) & (F.col("Price_Trend_20") == "Bullish"), "ACCUMULATE")
         .when((F.col("MACD_Line") < F.col("MACD_Signal")) & (F.col("Price_Trend_20") == "Bearish"), "DISTRIBUTE")
         .otherwise("HOLD")
    ) \
    .withColumn(
        "Confidence",
        F.round(
            (F.when(F.col("Price_Trend_20") == "Bullish", 1).otherwise(0) +
             F.when(F.col("Price_Trend_200") == "Uptrend", 1).otherwise(0) +
             F.when(F.col("RSI_14").between(30, 70), 1).otherwise(0) +
             F.when(F.col("MACD_Line") > F.col("MACD_Signal"), 1).otherwise(0)) / 4.0 * 100,
            2
        )
    )
 
gold_signals_table = f"{catalog}.{schema}.gold_trading_signals"
gold_signals.write.format("delta").mode("overwrite").saveAsTable(gold_signals_table)
 
print(f"✅ Created: {gold_signals_table}")